In [1]:
CLASSES_FILE = './data/classes.txt'
CONCEPTS_FILE = './data/concepts.txt'
with open(CLASSES_FILE, 'r') as f: 
    cub_classes = (f.read()).split('\n')

cub_classes = [c.strip().lower() for c in cub_classes]
print(cub_classes)


with open(CONCEPTS_FILE, 'r') as f: 
    cub_concept_names = (f.read()).split('\n')

cub_concept_names = [c.strip().lower() for c in cub_concept_names]
print(cub_concept_names)

['black footed albatross', 'laysan albatross', 'sooty albatross', 'groove billed ani', 'crested auklet', 'least auklet', 'parakeet auklet', 'rhinoceros auklet', 'brewer blackbird', 'red winged blackbird', 'rusty blackbird', 'yellow headed blackbird', 'bobolink', 'indigo bunting', 'lazuli bunting', 'painted bunting', 'cardinal', 'spotted catbird', 'gray catbird', 'yellow breasted chat', 'eastern towhee', 'chuck will widow', 'brandt cormorant', 'red faced cormorant', 'pelagic cormorant', 'bronzed cowbird', 'shiny cowbird', 'brown creeper', 'american crow', 'fish crow', 'black billed cuckoo', 'mangrove cuckoo', 'yellow billed cuckoo', 'gray crowned rosy finch', 'purple finch', 'northern flicker', 'acadian flycatcher', 'great crested flycatcher', 'least flycatcher', 'olive sided flycatcher', 'scissor tailed flycatcher', 'vermilion flycatcher', 'yellow bellied flycatcher', 'frigatebird', 'northern fulmar', 'gadwall', 'american goldfinch', 'european goldfinch', 'boat tailed grackle', 'eared 

In [ ]:
from CQA.datasets import GenericDataset

data = GenericDataset(ds_name='cub', split='train')

In [ ]:
# Load shapes3d images and CLIP embeddings
import torch
import os

device = 'cuda:2'
ds_name = 'cub'
#MODEL_NAME = os.environ.get('VLM', 'ViT-L/14')
MODEL_NAME = os.environ.get('VLM', 'ViT-L/14')
MODEL_NAME_SANITIZED = MODEL_NAME.replace('/','%')

activations_folder = os.path.join("/home/nicola.debole/projects/Caching","activations")
kwargs = {}
#check_activations(MODEL_NAME, ds_name, activations_folder, **kwargs)

with torch.no_grad():
    train_embeddings = torch.load(f'/home/nicola.debole/projects/Caching/activations/{ds_name}_train_{MODEL_NAME_SANITIZED}.pth', map_location=device)
    # Normalization
    train_embeddings = (train_embeddings - torch.mean(train_embeddings, dim=0, keepdim=True))/torch.std(train_embeddings, dim=0, keepdim=True)
    
    test_embeddings = torch.load(f'/home/nicola.debole/projects/Caching/activations/{ds_name}_test_{MODEL_NAME_SANITIZED}.pth', map_location=device)
    # Normalization
    test_embeddings = (test_embeddings - torch.mean(test_embeddings, dim=0, keepdim=True))/torch.std(test_embeddings, dim=0, keepdim=True)
    
    val_embeddings = torch.load(f'/home/nicola.debole/projects/Caching/activations/{ds_name}_val_{MODEL_NAME_SANITIZED}.pth', map_location=device)
    # Normalization
    val_embeddings = (val_embeddings - torch.mean(val_embeddings, dim=0, keepdim=True))/torch.std(val_embeddings, dim=0, keepdim=True)
    
from CQA.datasets import GenericDataset

data = GenericDataset(ds_name=ds_name, split='train')

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
from sklearn.svm import SVC, LinearSVC

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

model = MultiOutputClassifier(
    #LogisticRegression(max_iter=1000, solver='lbfgs')
    SVC(kernel='rbf', C=1.0, class_weight='balanced') #0.92
    #SVC(kernel='rbf', C=1000.0, class_weight='balanced') #0.90
    #SVC(kernel='linear', C=1.0) #0.87 
    #SVC(kernel='linear', C=0.01) #0.80
)

#concept_mask = [23,43,44,48,69,89]
concept_mask = [23,44,48,69,89,103]

#sparrow_ids = [i + 1 for i, name in enumerate(cub_classes) if 'sparrow' not in name]
#subset_concepts = list(df[df["classes_ids"].isin(sparrow_ids)].index)
subset_concepts = list(df[~df["classes_ids"].isin([123, 126])].index)
#subset_concepts = range(len(train_embeddings))
test_subset = np.where(np.isin(test_y, [123, 126]))[0]

print(train_concepts[subset_concepts].shape)
# 4. Train the model
model.fit(train_embeddings[subset_concepts].cpu(), train_concepts[subset_concepts][:, concept_mask])

print("Trained!")
# 5. Evaluate
concept_preds = model.predict(test_embeddings[test_subset].cpu())
print(classification_report(test_input_concepts[test_subset][:,concept_mask], concept_preds, target_names=[cub_concept_names[i] for i in concept_mask]))

In [ ]:
import numpy as np
import pandas as pd

csv_path = "/home/nicola.debole/projects/usty/usty/data/cub_csv.csv"
df = pd.read_csv(csv_path)

subset = df[df["classes_ids"].isin([123, 126])]

metadata_cols = ["classes_ids", "classes_names"]
concept_cols = [col for col in df.columns if col not in metadata_cols]

input_concepts = subset[concept_cols].to_numpy()
labels = subset["classes_ids"].to_numpy()
class_names = subset["classes_names"].to_numpy()

print(f"Subset size: {len(subset)}")
print(f"Classes: {np.unique(labels)}")
print(f"input_concepts shape: {input_concepts.shape}")
clf = LogisticRegression(max_iter=1000, class_weight='balanced') # clf = LogisticRegression(max_iter=1000, class_weight='balanced', penalty='elasticnet', solver='saga', l1_ratio=1)
transformed_concepts = 2*input_concepts[:,concept_mask] - 1

clf.fit(transformed_concepts, labels)

# Subset test to classes 123 and 126
test_mask = np.isin(test_y, [123, 126])
test_input_concepts_bin = test_input_concepts[test_mask]
test_y_bin = test_y[test_mask]
label_preds = clf.predict(2*test_input_concepts_bin[:,concept_mask]-1)

print(classification_report(test_y_bin, label_preds))

In [ ]:
# Predict the concepts
logits = [estimator.decision_function(test_embeddings.cpu()[test_mask]) for estimator in model.estimators_]
logits_array = np.column_stack(logits)
logits_transformed = np.tanh(logits_array)
# Predict the label
#print(logits_array)
# Get class prediction
y_pred = clf.predict(logits_transformed)
print(y_pred)
print(f"\n=== END-TO-END (TEST-SET) ===")

print(classification_report(test_y_bin, y_pred))
weights = clf.coef_[0]
print(weights)

In [ ]:
"""
concept_activations_df.py

Builds tidy DataFrames of predicted concept activations vs ground truth
for both train and test splits, then saves them to CSV.

Assumes the following variables are already in scope (from your existing script):
    model                  - fitted MultiOutputClassifier
    train_embeddings       - torch.Tensor  (N_train, D)
    test_embeddings        - torch.Tensor  (N_test,  D)
    input_concepts         - torch.Tensor  (N_train, C)  ground truth, train
    test_input_concepts    - torch.Tensor  (N_test,  C)  ground truth, test
    cub_concept_names      - list[str]     all concept names (indexed by concept_mask)
    concept_mask           - list[int]     indices of active concepts
    train_y                - torch.Tensor  (N_train,)    class labels, train
    test_y                 - torch.Tensor  (N_test,)     class labels, test
    cub_classes            - list[str]     class name for each label index
"""

import pandas as pd
import numpy as np

for i in concept_mask:
    print(f"Concept {i}: {cub_concept_names[i]}")
    
# ── 1. concept names for the active mask ─────────────────────────────────────
masked_concept_names = [cub_concept_names[i] for i in concept_mask]

# ── 2. predictions ────────────────────────────────────────────────────────────
#test_preds  = model.predict(test_embeddings[test_mask].cpu())    # (N_test,  len(concept_mask))
logits = [estimator.decision_function(test_embeddings.cpu()[test_mask]) for estimator in model.estimators_]
logits_array = np.column_stack(logits)
logits_transformed = np.tanh(logits_array)

# ── 3. ground truth numpy arrays ─────────────────────────────────────────────
test_gt  = test_input_concepts[test_mask][:, concept_mask].cpu().numpy()

# ── 4. class label arrays ────────────────────────────────────────────────────
test_labels  = test_y.cpu().numpy()[test_mask]
print(test_labels)
test_species  = [cub_classes[l] for l in test_labels]

# ── 5. helper: build one tidy dataframe ───────────────────────────────────────
def build_df(gt: np.ndarray,
             preds: np.ndarray,
             labels: np.ndarray,
             y_preds,
             species: list,
             concept_names: list,
             original_idx,
             split: str) -> pd.DataFrame:
    """
    Returns a DataFrame with columns:
        split | sample_idx | label | species
        | <concept>_gt | <concept>_pred | <concept>_correct  ...
    """
    n = gt.shape[0]

    # base columns
    base = pd.DataFrame({
        "split":      split,
        "sample_idx": original_idx,
        "label":      labels,
        "species":    species,
        "task_pred": y_preds
    })

    # ground truth columns
    gt_df = pd.DataFrame(
        gt,
        columns=[f"{c}_gt" for c in concept_names]
    )

    # prediction columns
    pred_df = pd.DataFrame(
        preds,
        columns=[f"{c}_pred" for c in concept_names]
    )

    # correct/incorrect per concept  (1 = correct, 0 = wrong)
    correct = (gt == preds).astype(int)
    correct_df = pd.DataFrame(
        correct,
        columns=[f"{c}_correct" for c in concept_names]
    )

    return pd.concat([base, gt_df, pred_df, correct_df], axis=1)

# ── 6. build dataframes ───────────────────────────────────────────────────────
test_df  = build_df(test_gt,  logits_transformed, test_labels, y_pred, test_species,
                    masked_concept_names, np.where(test_mask)[0], split="test")


print(test_df.head())

# ── 9. save to CSV ────────────────────────────────────────────────────────────
test_df.to_csv("/home/nicola.debole/projects/usty/usty/data/cub_user_study.csv",     index=False)


In [ ]:
dataframe = pd.read_csv('/home/nicola.debole/projects/usty/usty/data/cub_user_study.csv')

#betas = [ 0.70998989, -0.70998989,  0.70998989, -0.70998989, 0.70998989, -0.70998989]

betas = [ 0.70998989, -0.70998989,  0.70998989, -0.70998989, -0.70998989,  0.70998989]
df_predictions = dataframe['task_pred']

concept_list = []
for i in concept_mask:
    concept_list.append(f"{cub_concept_names[i]}_pred")
print(concept_list)

p_concepts = dataframe[concept_list]

def logistic(x):
    return 1 / (1 + np.exp(-x))

def computePrediction(activations):
    eta = 0 #; // Add intercept here if needed: eta = YOUR_INTERCEPT;
    for i in range(6):
        eta += betas[i] * activations.iloc[:,i]
    return logistic(eta)

toy_model = computePrediction(p_concepts)
toy_predictions = (toy_model > 0.5).astype(int)
toy_predictions = np.where(toy_predictions == 0, 123, 126)
#print(toy_predictions)
#print(df_predictions)

diff = toy_predictions != df_predictions
diff_list = np.where(diff)[0]
print(diff_list)
print('Differences in whole dataset', np.sum(diff))